# Preliminary Study — Thematic Analysis of Semi-Structured Interviews

This notebook performs the thematic analysis on the double-coded interview transcripts from the 12 semi-structured interviews with DS practitioners conducted as the preliminary study.

**Methodology:** Reflexive thematic analysis following the guidelines of Braun and Clarke (2006).
**Double coding:** Each transcript was independently coded by two coders. The `All-Sum` sheet in `coding_results.xlsx` contains the consolidated counts across all 12 participants for every code, broken down by interview question.

**Analysis steps:**
1. Load the `All-Sum` consolidated sheet from `coding_results.xlsx`
2. Extract codes and their per-question mention counts
3. Retain every code whose total mention count is greater than zero
4. Group the retained codes into an initial set of themes for review

In [1]:
!pip install openpyxl pandas


[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 1. Setup

In [2]:
import openpyxl
import pandas as pd
from collections import defaultdict

XLSX_PATH = "coding_results.xlsx"
SHEET_NAME = "All-Sum"
HEADER_ROW = 11        # row containing question column headers
DATA_START_ROW = 12    # first row of code data
NUM_QUESTION_COLS = 12 # columns 1..12 hold the 12 question counts

## 2. Load the All-Sum Sheet

The `All-Sum` sheet aggregates the consensus counts from all 12 participant `<Name>-Sum` sheets. Each row is a code; columns 1–12 hold the per-question mention counts summed across participants.

In [3]:
wb = openpyxl.load_workbook(XLSX_PATH, data_only=True)
assert SHEET_NAME in wb.sheetnames, f"Expected sheet '{SHEET_NAME}' in workbook"
ws = wb[SHEET_NAME]
print(f"Loaded sheet '{SHEET_NAME}' (rows={ws.max_row}, cols={ws.max_column})")

Loaded sheet 'All-Sum' (rows=735, cols=231)


## 3. Define Interview Question Labels

Short English labels for the 12 question columns.

In [4]:
Q_LABELS = [
    "Q1: DS process awareness",
    "Q1a: Time per DS phase",
    "Q2: DS challenges",
    "Q3: SE practices used",
    "Q3a: Benefits of SE practices",
    "Q4: Future SE practices",
    "Q5: Process/tool changes over time",
    "Q6: Tools used",
    "Q6a: Specific tools",
    "Q6b: Tools per DS phase",
    "Q6c: Tool challenges",
    "Q7: Dream tool",
]
assert len(Q_LABELS) == NUM_QUESTION_COLS

## 4. Extract All Codes With Total Count Greater Than Zero

We iterate over every row from `DATA_START_ROW` onward. Rows whose total across the 12 question columns is zero are visual section-header rows in the spreadsheet and are excluded. Every code with at least one mention is retained — there is no further inclusion threshold.

In [5]:
# code_name -> {total_mentions, mentions_by_question}
code_stats = {}

for row in ws.iter_rows(min_row=DATA_START_ROW, max_row=ws.max_row, values_only=True):
    code_name = row[0]
    if code_name is None:
        continue
    counts = [v if isinstance(v, (int, float)) else 0 for v in row[1:1 + NUM_QUESTION_COLS]]
    total = sum(counts)
    if total <= 0:
        # section header / separator row in the spreadsheet
        continue
    by_q = {Q_LABELS[i]: int(c) for i, c in enumerate(counts) if c > 0}
    # If a code name appears more than once, merge counts (additive)
    if code_name in code_stats:
        code_stats[code_name]["total_mentions"] += int(total)
        for q, c in by_q.items():
            code_stats[code_name]["mentions_by_question"][q] = (
                code_stats[code_name]["mentions_by_question"].get(q, 0) + c
            )
    else:
        code_stats[code_name] = {
            "total_mentions": int(total),
            "mentions_by_question": dict(by_q),
        }

print(f"Total unique codes with count > 0: {len(code_stats)}")

Total unique codes with count > 0: 79


## 5. Display the Retained Codes

In [6]:
df_retained = (
    pd.DataFrame([
        {"Code": code, "Total mentions": stats["total_mentions"]}
        for code, stats in code_stats.items()
    ])
    .sort_values(["Total mentions", "Code"], ascending=[False, True])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
df_retained

,Code,Total mentions
0,Data preparation,4
1,AWS,3
2,Aware but do not use any specific framework,3
3,CRISP-DM,3
4,Jira,3
5,Python libraries,3
6,Stand-up meeting,3
7,Auto visualization,2
8,Git,2
9,Modeling,2


## 6. Initial Thematic Grouping

The codes retained above were sorted, compared, and grouped into an initial set of themes following the inductive phase of reflexive thematic analysis. The groupings below are a **first pass for review** — codes can be moved between themes or sub-themes, themes can be split or merged, and labels can be revised during the next reading.

Themes that contain a natural internal split are organized into **sub-themes** (e.g., DS challenges, SE practices adopted, Tools used). Themes without an internal split use a single placeholder key (`"_"`).

In [7]:
# Themes are nested: theme -> sub-theme -> [codes]
# Use a single placeholder sub-theme name "_" when the theme has no sub-themes.

THEME_MAP = {
    # T1 — DS process awareness & framework adoption (no sub-themes)
    "T1: DS process awareness & framework adoption": {
        "_": [
            "Aware but do not use any specific framework",
            "CRISP-DM",
        ],
    },

    # T2 — DS lifecycle phases & effort distribution (no sub-themes)
    "T2: DS lifecycle phases & effort distribution": {
        "_": [
            "Business understanding",
            "Data preparation",
            "Data cleansing",
            "Modeling",
            "Evaluation",
            "Deployment",
        ],
    },

    # T3 — DS challenges
    "T3: DS challenges": {
        "T3a: Data quality, drift & evaluation": [
            "Data inconsistency or data behavior shift or data drift",
            "Data assertion problem and data observability and monitoring",
            "Lengthy AB testing",
            "Cannot evaluate model due to modified data (hashed)",
        ],
        "T3b: Requirements, communication & maintainability": [
            "Misunderstanding between DS team and users",
            "Many stakeholders",
            "Requirement changes",
            "Requirement gathering",
            "No documentation",
            "Analytic engineering",
            "DBT",
        ],
    },

    # T4 — SE practices adopted
    "T4: SE practices adopted": {
        "T4a: Agile coordination": [
            "Agile",
            "Scrum or Kanban board",
            "Stand-up meeting",
            "Short releases or sprints",
            "Get team's suggestion",
            "Help following the plan",
            "Update progress with the team",
            "Use all 7 SE steps",
            "Requirements analysis and design",
        ],
        "T4b: Testing, quality & version control": [
            "Integration testing",
            "Model testing",
            "UAT",
            "Exit criteria",
            "Pre-commit",
            "SonarQube",
            "Version control",
        ],
    },

    # T5 — Future / desired SE practices (no sub-themes)
    "T5: Future / desired SE practices": {
        "_": [
            "Cloud services",
            "DevOps (include CI-CD)",
            "Requirement gathering patterns",
            "maintain code quality",
        ],
    },

    # T6 — Tools (merged from previous T6 + T7 + T8)
    "T6: Tools — usage, challenges & dream tools": {
        "T6a: Languages, notebooks & IDEs": [
            "Python",
            "R",
            "SAS",
            "Scala",
            "Python libraries",
            "Jupyter Notebook",
            "Jupyter lab",
            "pycharm",
            "CDSW Workbench",
            "Local development",
        ],
        "T6b: Cloud, data engineering & model serving": [
            "AWS",
            "EC2",
            "SageMaker",
            "Big Query",
            "Hadoop",
            "Databricks",
            "Dataflow",
            "Airflow",
            "Coltjob",
            "Streamlit",
            "Tensorflowserving",
            "Torchserve",
            "Predefine models",
            "Tableau",
        ],
        "T6c: Project management & collaboration": [
            "Git",
            "Git repository",
            "Jira",
            "Kanban board",
            "Blacklink",
            "I Source",
        ],
        "T6d: Tool challenges": [
            "Connections between services from different vendors",
            "Not using tools",
        ],
        "T6e: Dream tools — automation, monitoring & integration": [
            "Auto visualization",
            "Detection of data problems",
            "Data quality measurement",
            "Guarantee data behavior after data governance or PDPA",
            "DORA metrics for data products",
            "Integrating the DS project with client systems",
            "Collaboration tool",
            "Unrestricted tool use (AWS) without the organization restrictions",
        ],
    },
}

# Helper: iterate (theme, sub_theme_or_None, code) tuples
def iter_theme_entries(theme_map):
    for theme, sub_map in theme_map.items():
        for sub, codes in sub_map.items():
            sub_label = None if sub == "_" else sub
            for code in codes:
                yield theme, sub_label, code

# Helper: flatten codes for a theme across sub-themes
def codes_in_theme(sub_map):
    return [c for codes in sub_map.values() for c in codes]


## 7. Thematic Map — Summary Table

The table below lists every theme together with its constituent codes, mention totals, and a coverage flag indicating whether the code was actually found in the All-Sum sheet. Codes flagged ✗ are placeholders to revisit (typically caused by a label mismatch with the spreadsheet).

In [8]:
rows = []
for theme, sub_label, code in iter_theme_entries(THEME_MAP):
    stats = code_stats.get(code)
    rows.append({
        "Theme": theme,
        "Sub-theme": sub_label if sub_label else "",
        "Code": code,
        "Total mentions": stats["total_mentions"] if stats else 0,
        "Found in All-Sum": "✓" if stats else "✗",
    })

df_map = pd.DataFrame(rows)
df_map

,Theme,Sub-theme,Code,Total mentions,Found in All-Sum
0,T1: DS process awareness & framework adoption,,Aware but do not use any specific framework,3,✓
1,T1: DS process awareness & framework adoption,,CRISP-DM,3,✓
2,T2: DS lifecycle phases & effort distribution,,Business understanding,1,✓
3,T2: DS lifecycle phases & effort distribution,,Data preparation,4,✓
4,T2: DS lifecycle phases & effort distribution,,Data cleansing,1,✓
5,T2: DS lifecycle phases & effort distribution,,Modeling,2,✓
6,T2: DS lifecycle phases & effort distribution,,Evaluation,1,✓
7,T2: DS lifecycle phases & effort distribution,,Deployment,1,✓
8,T3: DS challenges,"T3a: Data quality, drift & evaluation",Data inconsistency or data behavior shift or data drift,1,✓
9,T3: DS challenges,"T3a: Data quality, drift & evaluation",Data assertion problem and data observability and monitoring,1,✓


## 8. Per-Theme Summary

In [9]:
for theme, sub_map in THEME_MAP.items():
    all_codes = codes_in_theme(sub_map)
    theme_total = sum(code_stats.get(c, {}).get("total_mentions", 0) for c in all_codes)
    matched = sum(1 for c in all_codes if c in code_stats)
    print(f"\n{'=' * 78}")
    print(f"  {theme}")
    print(f"  ({matched}/{len(all_codes)} codes matched, total mentions = {theme_total})")
    print('=' * 78)
    for sub, codes in sub_map.items():
        if sub != "_":
            sub_total = sum(code_stats.get(c, {}).get("total_mentions", 0) for c in codes)
            sub_matched = sum(1 for c in codes if c in code_stats)
            print(f"\n  -- {sub}  ({sub_matched}/{len(codes)} codes, {sub_total} mentions) --")
        for code in codes:
            stats = code_stats.get(code)
            if stats is None:
                print(f"    ✗  {code}  - not found in All-Sum")
                continue
            t = stats["total_mentions"]
            by_q = stats["mentions_by_question"]
            print(f"    ✓  {code}  (total={t})")
            for q, cnt in sorted(by_q.items(), key=lambda x: -x[1]):
                print(f"          {q}: {cnt}")


  T1: DS process awareness & framework adoption
  (2/2 codes matched, total mentions = 6)
    ✓  Aware but do not use any specific framework  (total=3)
          Q1: DS process awareness: 2
          Q1a: Time per DS phase: 1
    ✓  CRISP-DM  (total=3)
          Q1: DS process awareness: 3

  T2: DS lifecycle phases & effort distribution
  (6/6 codes matched, total mentions = 10)
    ✓  Business understanding  (total=1)
          Q1a: Time per DS phase: 1
    ✓  Data preparation  (total=4)
          Q1a: Time per DS phase: 4
    ✓  Data cleansing  (total=1)
          Q1a: Time per DS phase: 1
    ✓  Modeling  (total=2)
          Q1a: Time per DS phase: 2
    ✓  Evaluation  (total=1)
          Q1a: Time per DS phase: 1
    ✓  Deployment  (total=1)
          Q1a: Time per DS phase: 1

  T3: DS challenges
  (11/11 codes matched, total mentions = 11)

  -- T3a: Data quality, drift & evaluation  (4/4 codes, 4 mentions) --
    ✓  Data inconsistency or data behavior shift or data drift  (tot

## 9. Codes Not Yet Assigned to a Theme

Any code with a positive count that does not appear in `THEME_MAP` is listed here so it can be slotted into an existing theme or used to seed a new one during the review pass.

In [10]:
assigned = {code for _, _, code in iter_theme_entries(THEME_MAP)}
unassigned = sorted(c for c in code_stats if c not in assigned)

if unassigned:
    df_unassigned = pd.DataFrame([
        {"Code": c, "Total mentions": code_stats[c]["total_mentions"]}
        for c in unassigned
    ]).sort_values(["Total mentions", "Code"], ascending=[False, True]).reset_index(drop=True)
else:
    df_unassigned = pd.DataFrame(columns=["Code", "Total mentions"])

print(f"Unassigned codes: {len(df_unassigned)}")
df_unassigned

Unassigned codes: 0


,Code,Total mentions
